# Spect Week 1 — Assisted Speculative Decoding (Colab T4)

**Draft:** `Qwen/Qwen2.5-0.5B-Instruct`  
**Target:** `Qwen/Qwen2.5-3B-Instruct` (4-bit)  

Runtime → Change runtime type → **T4 GPU** before running.

Optional: set `USE_7B = True` to try `Qwen2.5-7B-Instruct` 4-bit.

In [ ]:
# Optional: clone your repo. Replace with your GitHub URL if pushed.
# Or upload the Spect folder / mount Drive.
import os
from pathlib import Path

REPO_URL = os.environ.get("SPECT_REPO_URL", "")  # e.g. https://github.com/you/Spect.git
WORKDIR = Path("/content/Spect")

if REPO_URL and not WORKDIR.exists():
    !git clone {REPO_URL} {WORKDIR}
elif not WORKDIR.exists():
    print("Upload/copy the Spect project to /content/Spect or set SPECT_REPO_URL")

%cd /content/Spect
!pip install -q -e ".[gpu]"

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
USE_7B = False  # True -> Qwen2.5-7B-Instruct 4-bit
K_SWEEP = True  # K = 2, 4, 8 against one baseline. False uses configs/week1_qwen.yaml (K=5).
MAX_NEW_TOKENS = 128
LIMIT = None  # e.g. 2 for a quick smoke run

import importlib
from pathlib import Path

import spect.benchmark
import spect.decode
import spect.metrics

# Re-run picks up source edits without a stale imported generate().
importlib.reload(spect.metrics)
importlib.reload(spect.decode)
importlib.reload(spect.benchmark)
from spect.benchmark import load_config, run_benchmark

root = Path("/content/Spect").resolve()
if USE_7B:
    cfg_name = "configs/week1_qwen_7b.yaml"
elif K_SWEEP:
    cfg_name = "configs/week1_qwen_k_sweep.yaml"
else:
    cfg_name = "configs/week1_qwen.yaml"
config = load_config(root / cfg_name)

out = run_benchmark(
    config,
    project_root=root,
    device="cuda",
    max_new_tokens=MAX_NEW_TOKENS,
    limit=LIMIT,
    use_fallback_target=False,
)
out["summary"]

In [ ]:
import json
from pathlib import Path

result_path = Path(out["path"])
payload = json.loads(result_path.read_text())
summary = payload["summary"]

if "by_k" in summary:
    print(f"Best K={summary['best_k']}  speedup={summary['speedup']}x")
    for k, block in summary["by_k"].items():
        print(
            f"K={k}  speedup={block['speedup']}x  "
            f"accept={block['acceptance_rate']}  "
            f"mean_len={block['mean_accepted_length']}  "
            f"match={block['token_match_rate']}  "
            f"≥1.3x={block['meets_week1_target_1_3x']}"
        )
else:
    print(f"Speedup: {summary['speedup']}x")
    print(f"Meets ≥1.3x: {summary['meets_week1_target_1_3x']}")
    print(f"Acceptance: {summary.get('acceptance_rate')}  match: {summary.get('token_match_rate')}")

print("\nPer prompt:")
for r in payload["runs"]:
    if r["mode"] != "speculative":
        continue
    ex = r["extra"]
    print(
        f"{r['prompt_id']:16s} K={ex.get('assistant_num_candidate_tokens')}  "
        f"{r['tokens_per_sec']:.2f} tok/s  "
        f"match={ex.get('tokens_match')}  "
        f"accept={ex.get('acceptance_rate')}  "
        f"mean_len={ex.get('mean_accepted_length')}"
    )

## Notes

- Default target is **3B** for reliable free-T4 runs.
- Greedy decoding (`temperature=0`) keeps assisted generation comparable to baseline.
- Download results JSON from `results/` for your writeup.
- Week 2 will add a live metrics dashboard on top of this JSON schema.